# SmolVLA · 0.45B 小模型推理一次

**这个 notebook 在做什么**：加载 SmolVLA 的 LIBERO 微调 checkpoint，闭环推理一次并录下成功视频。SmolVLA 是"把 VLA 做小"的代表：SmolVLM2 底座 + flow matching 动作专家，总共 0.45B 参数，消费级显卡就能跑。

**看点**：① 加载与调用接口和 π0/π0.5 完全一致；② 模型小一个数量级，加载和每步推理都明显更快——对比 4_1 的 7B OpenVLA 体感强烈；③ 异步推理（讲义 §7.4）没有出现在这段代码里：它是把这套 `select_action` 搬到 PolicyServer 的部署层改造，模型本身不用改。

> 在 `code/` 目录启动 Jupyter 内核运行；需要 GPU 与 `uv sync --extra gpu_x86` 环境，权重从 HF Hub 自动下载（落 `$HF_HOME`）。

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import torch

from lerobot.configs.policies import PreTrainedConfig
from lerobot.envs.configs import LiberoEnv as LiberoEnvConfig
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.envs.utils import add_envs_task, preprocess_observation
from lerobot.policies.factory import get_policy_class, make_pre_post_processors
from lerobot.utils.io_utils import write_video
import lerobot.policies  # noqa: F401

# MuJoCo 的离屏渲染需要 EGL。
os.environ.setdefault("MUJOCO_GL", "egl")

# 关闭 torch compile / inductor，避免首次运行时出现大量 autotune 开销，
# 让课堂 demo 更稳定、更可复现。
os.environ.setdefault("TORCHINDUCTOR_DISABLE", "1")
os.environ.setdefault("TORCH_COMPILE_DISABLE", "1")

# 下面这组参数是已经验证过能跑出 success=True 的固定配置。
# SmolVLA：0.45B 的社区小模型路线（SmolVLM2 底座 + flow matching 动作专家），
# 消费级显卡就能推理，是“把 VLA 做小”的代表。
POLICY_PATH = "lerobot/smolvla_libero"
TASK_SUITE = "libero_goal"
TASK_ID = 5
EPISODE_INDEX = 2
MAX_STEPS = 180
FPS = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 7
OUT_PATH = Path("vla/4_vla_inference/4_4_smolvla_infer/output/smolvla_libero_success.mp4")


## 1 固定初始状态

LIBERO 每个任务有多个初始状态；把 `init_state_id` 锁定，实验才可复现——你换任何超参重跑，对照的都是同一局。

In [ ]:
def set_episode_index(env, episode_index: int) -> None:
    # LeRobot 的 LIBERO 向量环境外面包了一层 SyncVectorEnv。
    # 真正控制初始状态的是里面每个子环境的 episode_index / init_state_id。
    # 这里只跑 1 个环境，所以直接把第 0 个子环境切到我们选好的成功初始状态。
    for inner_env in env.envs:
        inner_env.episode_index = episode_index
        inner_env.init_state_id = episode_index


## 2 闭环主循环

与 π0 demo 逐行同构。值得注意的是这已经是本组第 5 个模型共用同一段闭环代码——**策略即函数**的抽象让"换模型"变成换一行 `POLICY_PATH`，这正是讲 8 建立的 policy 闭环直觉在模型导览里的回报。

In [ ]:
def main() -> None:
    # 固定随机种子，保证每次讲课演示时拿到相同的 rollout。
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    # 1. 读取 Hugging Face 上保存的 policy 配置。
    # 2. 把 device 改成当前机器可用的 cuda / cpu。
    # 3. 加载策略本体和它对应的 preprocess / postprocess 流水线。
    # strict=False 是为了兼容当前 lerobot 版本和权重里少量 buffer 命名差异。
    policy_cfg = PreTrainedConfig.from_pretrained(POLICY_PATH)
    policy_cfg.device = DEVICE
    policy = get_policy_class(policy_cfg.type).from_pretrained(POLICY_PATH, config=policy_cfg, strict=False)
    preprocessor, postprocessor = make_pre_post_processors(policy_cfg, pretrained_path=POLICY_PATH)

    # 构建 LIBERO 环境。这里只保留和成功案例匹配的最小参数。
    env_cfg = LiberoEnvConfig(
        task=TASK_SUITE,
        task_ids=[TASK_ID],
        obs_type="pixels_agent_pos",
        observation_height=256,
        observation_width=256,
        episode_length=MAX_STEPS,
    )
    env = make_env(env_cfg, n_envs=1)[TASK_SUITE][TASK_ID]
    print(f"task: {env.envs[0].task} | instruction: {env.envs[0].task_description}")
    env_preprocessor, env_postprocessor = make_env_pre_post_processors(env_cfg, policy_cfg)

    frames = []
    success = False

    try:
        policy.reset()

        # 切换到底层 LIBERO 子环境里已经验证可成功的 init state。
        set_episode_index(env, EPISODE_INDEX)
        observation, _ = env.reset(seed=[SEED + EPISODE_INDEX])

        for _ in range(MAX_STEPS):
            # 环境原始 observation 先转成 LeRobot 约定的扁平 key 格式，
            # 再补上 task 文本，随后送进 env processor 和 policy processor。
            observation_batch = preprocess_observation(observation)
            observation_batch = add_envs_task(env, observation_batch)
            observation_batch = env_preprocessor(observation_batch)
            observation_batch = preprocessor(observation_batch)

            # SmolVLA 每次用 flow matching 动作专家去噪出一个动作块，
            # select_action 内部维护动作队列：块没用完时直接出队，不再过模型。
            with torch.inference_mode():
                action = policy.select_action(observation_batch)

            # postprocessor 负责把 policy 输出还原回环境动作空间。
            action = postprocessor(action)
            action = env_postprocessor({"action": action})["action"].cpu().numpy()

            # 执行动作，并把渲染帧缓存下来，最后统一写 mp4。
            observation, _, terminated, truncated, info = env.step(action)
            frames.append(env.envs[0].render())

            # LIBERO 的 success 信号放在 final_info 里。
            if "final_info" in info and isinstance(info["final_info"], dict):
                success = bool(info["final_info"]["is_success"][0])

            if bool(terminated[0]) or bool(truncated[0]):
                break

        # 这两个检查保证 demo 不是“看起来运行了”，而是真的有结果、而且真的成功。
        if not frames:
            raise RuntimeError("no frames")
        if not success:
            raise RuntimeError("no success")

        # 把整段 rollout 直接导出成 mp4。
        write_video(str(OUT_PATH), frames, fps=FPS)
        print(OUT_PATH)
    finally:
        # 关闭环境，避免 MuJoCo / EGL 资源泄漏。
        env.close()


if __name__ == "__main__":
    main()
